# POC: HRDEM Coverage Layer 4 BBox Download (EPSG:3978)

This notebook reads the same `lores.tif`, reprojects its bbox to EPSG:3978,
queries NRCan HRDEM coverage layer 4 with server-side native projection,
and writes the returned GeoJSON to `_scratch`.


## 01) Imports And Constants

Keep steps explicit and log key parameters for reproducibility.


In [1]:
# Standard library imports for timing, paths, and API inspection.
import inspect
import json
import os
import time
from pathlib import Path

# Third-party imports for raster bbox handling and HTTP requests.
import rasterio
import requests
from rasterio.warp import transform_bounds


In [2]:
# Print package versions from this runtime.
print(f"rasterio: {rasterio.__version__}")
print(f"requests: {requests.__version__}")


rasterio: 1.5.0
requests: 2.32.5


In [3]:
# Hard-code notebook cwd to project root and start runtime clock.
PROJECT_ROOT = Path("/workspace").resolve()
assert PROJECT_ROOT.exists(), f"Missing expected project root: {PROJECT_ROOT}"
os.chdir(PROJECT_ROOT)
assert Path.cwd() == PROJECT_ROOT, f"Unexpected cwd: {Path.cwd()}"
notebook_t0 = time.perf_counter()

# Input raster and output GeoJSON path.
LOWRES_FP = Path("tests/data/fathom_n51w115_clip/lores.tif")
assert LOWRES_FP.exists(), f"Missing low-res tile: {LOWRES_FP.resolve()}"
OUT_GEOJSON_FP = Path("_scratch") / "coverage_HRDEM_en_layer4_lores_bbox3978.geojson"

# ArcGIS REST layer endpoint and query endpoint.
LAYER_URL = "https://maps-cartes.services.geo.ca/server_serveur/rest/services/NRCan/coverage_HRDEM_en/MapServer/4"
QUERY_URL = f"{LAYER_URL}/query"
TARGET_QUERY_EPSG = 3978

print("Configured constants:")
print(f"  PROJECT_ROOT: {PROJECT_ROOT}")
print(f"  LOWRES_FP: {LOWRES_FP.resolve()}")
print(f"  OUT_GEOJSON_FP: {OUT_GEOJSON_FP.resolve()}")
print(f"  LAYER_URL: {LAYER_URL}")
print(f"  QUERY_URL: {QUERY_URL}")
print(f"  TARGET_QUERY_EPSG: {TARGET_QUERY_EPSG}")


Configured constants:
  PROJECT_ROOT: /workspace
  LOWRES_FP: /workspace/tests/data/fathom_n51w115_clip/lores.tif
  OUT_GEOJSON_FP: /workspace/_scratch/coverage_HRDEM_en_layer4_lores_bbox3978.geojson
  LAYER_URL: https://maps-cartes.services.geo.ca/server_serveur/rest/services/NRCan/coverage_HRDEM_en/MapServer/4
  QUERY_URL: https://maps-cartes.services.geo.ca/server_serveur/rest/services/NRCan/coverage_HRDEM_en/MapServer/4/query
  TARGET_QUERY_EPSG: 3978


## 02) Environment And API Surface Check

Confirm key API signatures from the active environment.


In [4]:
# Inspect key call signatures used in this notebook.
print("Key API signatures:")
print(f"  requests.get: {inspect.signature(requests.get)}")
print(f"  rasterio.open: {inspect.signature(rasterio.open)}")
print(f"  transform_bounds: {inspect.signature(transform_bounds)}")


Key API signatures:
  requests.get: (url, params=None, **kwargs)
  rasterio.open: (fp, mode='r', driver=None, width=None, height=None, count=None, crs=None, transform=None, dtype=None, nodata=None, sharing=False, thread_safe=False, opener=None, **kwargs)
  transform_bounds: (src_crs, dst_crs, left, bottom, right, top, densify_pts=21)


## 03) Read `lores.tif` And Build Query BBox In EPSG:3978

Use the low-res tile bounds and reproject to the service-native query projection.


In [5]:
# Read low-res raster bounds and transform them to EPSG:3978 for server-side filtering.
with rasterio.open(LOWRES_FP) as lowres_ds:
    lowres_crs = lowres_ds.crs
    lowres_bounds = tuple(lowres_ds.bounds)
    lowres_shape = (lowres_ds.height, lowres_ds.width)
    lowres_res = lowres_ds.res

bbox_3978 = transform_bounds(
    lowres_crs,
    f"EPSG:{TARGET_QUERY_EPSG}",
    *lowres_bounds,
    densify_pts=21,
)

# Keep a 4326 bbox for reference diagnostics only.
bbox_4326 = transform_bounds(
    lowres_crs,
    "EPSG:4326",
    *lowres_bounds,
    densify_pts=21,
)

assert lowres_crs is not None, "Expected low-res raster CRS to exist."
assert bbox_3978[0] < bbox_3978[2], "Invalid EPSG:3978 bbox x ordering."
assert bbox_3978[1] < bbox_3978[3], "Invalid EPSG:3978 bbox y ordering."

print("Low-res raster details:")
print(f"  CRS: {lowres_crs}")
print(f"  Bounds ({lowres_crs}): {lowres_bounds}")
print(f"  Shape (rows, cols): {lowres_shape}")
print(f"  Resolution: {lowres_res}")
print("\nQuery bbox:")
print(f"  EPSG:3978: {bbox_3978}")
print(f"  EPSG:4326 (diagnostic): {bbox_4326}")


Low-res raster details:
  CRS: EPSG:4326
  Bounds (EPSG:4326): (-114.895639271, 51.36125093, -114.084921156, 51.560328722)
  Shape (rows, cols): (717, 2919)
  Resolution: (0.0002777383059266873, 0.0002776538242677883)

Query bbox:
  EPSG:3978: (-1351706.1478516518, 457759.767268949, -1291837.2138732676, 495537.1407820382)
  EPSG:4326 (diagnostic): (-114.895639271, 51.36125093, -114.084921156, 51.560328722)


## 04) Query Layer 4 And Download GeoJSON

Request `f=geojson` with `geometry` envelope in EPSG:3978 so filtering runs server-side in native projection.


In [6]:
# Build ArcGIS query parameters with geometry in EPSG:3978.
xmin, ymin, xmax, ymax = bbox_3978
params = {
    "f": "geojson",
    "where": "1=1",
    "geometry": f"{xmin},{ymin},{xmax},{ymax}",
    "geometryType": "esriGeometryEnvelope",
    "inSR": str(TARGET_QUERY_EPSG),
    "outSR": str(TARGET_QUERY_EPSG),
    "spatialRel": "esriSpatialRelIntersects",
    "outFields": "*",
    "returnGeometry": "true",
}

# Execute timed request.
request_t0 = time.perf_counter()
response = requests.get(QUERY_URL, params=params, timeout=180)
request_seconds = time.perf_counter() - request_t0
response.raise_for_status()

# Parse payload and validate shape.
payload = response.json()
assert payload.get("type") == "FeatureCollection", "Expected GeoJSON FeatureCollection response."
feature_count = len(payload.get("features", []))
assert feature_count > 0, "Query returned zero features for lores bbox."

print("Query response summary:")
print(f"  HTTP status: {response.status_code}")
print(f"  Request seconds: {request_seconds:.3f}")
print(f"  Response bytes: {len(response.content):,}")
print(f"  Feature count: {feature_count}")
print(f"  Response CRS: {payload.get('crs')}")


Query response summary:
  HTTP status: 200
  Request seconds: 0.613
  Response bytes: 172,630
  Feature count: 3
  Response CRS: {'type': 'name', 'properties': {'name': 'EPSG:3978'}}


## 05) Write GeoJSON To `_scratch`

Persist the query result to local disk for downstream use.


In [7]:
# Write payload to disk and validate file output.
OUT_GEOJSON_FP.parent.mkdir(parents=True, exist_ok=True)
OUT_GEOJSON_FP.write_text(json.dumps(payload, indent=2))

assert OUT_GEOJSON_FP.exists(), f"Expected output file to exist: {OUT_GEOJSON_FP}"
assert OUT_GEOJSON_FP.stat().st_size > 0, "Output GeoJSON is empty."

first_property_keys = sorted(payload["features"][0].get("properties", {}).keys())[:10]
print("Wrote GeoJSON:")
print(f"  {OUT_GEOJSON_FP.resolve()}")
print(f"  Size (bytes): {OUT_GEOJSON_FP.stat().st_size:,}")
print(f"  First feature property keys: {first_property_keys}")


Wrote GeoJSON:
  /workspace/_scratch/coverage_HRDEM_en_layer4_lores_bbox3978.geojson
  Size (bytes): 400,058
  First feature property keys: ['OBJECTID', 'SHAPE_Area', 'SHAPE_Length', 'coord_sys1', 'coord_sys2', 'coord_sys3', 'ftp_dsm1', 'ftp_dsm2', 'ftp_dsm3', 'ftp_dtm1']


## 06) Runtime Summary

Report both query-time and full notebook elapsed time.


In [8]:
# Print final timing summary.
notebook_seconds = time.perf_counter() - notebook_t0
print("Final summary:")
print(f"  Layer URL: {LAYER_URL}")
print(f"  Query EPSG: {TARGET_QUERY_EPSG}")
print(f"  Query bbox EPSG:3978: {bbox_3978}")
print(f"  Features downloaded: {feature_count}")
print(f"  Output file: {OUT_GEOJSON_FP.resolve()}")
print(f"  Query request seconds: {request_seconds:.3f}")
print(f"  Notebook elapsed seconds: {notebook_seconds:.3f}")


Final summary:
  Layer URL: https://maps-cartes.services.geo.ca/server_serveur/rest/services/NRCan/coverage_HRDEM_en/MapServer/4
  Query EPSG: 3978
  Query bbox EPSG:3978: (-1351706.1478516518, 457759.767268949, -1291837.2138732676, 495537.1407820382)
  Features downloaded: 3
  Output file: /workspace/_scratch/coverage_HRDEM_en_layer4_lores_bbox3978.geojson
  Query request seconds: 0.613
  Notebook elapsed seconds: 0.736
